In [1]:
import kagglehub

path = kagglehub.dataset_download("rmisra/news-headlines-dataset-for-sarcasm-detection")



In [2]:
import json

with open(path + "/Sarcasm_Headlines_Dataset.json") as f:
    data = [json.loads(line) for line in f]

sentences = []
labels = []
urls = []
for item in data:
    sentences.append(item['headline'])
    labels.append(item['is_sarcastic'])
    urls.append(item['article_link'])

In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


I0000 00:00:1789158741.043580  263343 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789158741.044141  263343 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789158741.084243  263343 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789158742.044548  263343 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [4]:
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
# instancia o objeto com limite de palavras
tokenizer.fit_on_texts(sentences)
# transforma cada palavra em um indice dentro do vocabulario

In [5]:
word_index = tokenizer.word_index
# atribui a cada palavra um numero inteiro

In [6]:
sequences = tokenizer.texts_to_sequences(sentences)
# pega as frases e transforma-as em sequencias de indices de acordo com o vocabulario criado

In [7]:
padded = pad_sequences(sequences, padding='post', maxlen=120, truncating='post')
# para cada sequencia de indices, no caso de faltar tamnho para algum, será preenchido com 0s

In [8]:
print(padded[0])

[308   1 679   1   1  48 382   1   1   6   1   1   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0]


In [9]:
training_sentences = sentences[:20000]
testing_sentences = sentences[20000:]
training_labels = labels[:20000]
testing_labels = labels[20000:]

In [10]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(training_sentences)
# instancia o objeto com limite de palavras e ajusta para os dados de treinamento

In [11]:
word_index = tokenizer.word_index
# atribui a cada palavra um numero inteiro

In [12]:
training_sentences = tokenizer.texts_to_sequences(training_sentences)
training_padded = pad_sequences(training_sentences, padding='post', maxlen=120, truncating='post')
# transforma as frases de treinamento em sequencias de indices e preenche com zeros

In [13]:
testing_sentences = tokenizer.texts_to_sequences(testing_sentences)
testing_padded = pad_sequences(testing_sentences, padding='post', maxlen=120, truncating='post')
# transforma as frases de teste em sequencias de indices e preenche com zeros

In [14]:
import numpy as np

training_padded = np.array(training_padded)
testing_padded = np.array(testing_padded)
training_labels = np.array(training_labels)
testing_labels = np.array(testing_labels)

In [15]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(10000, 16, input_length=120),
    # o embedding atribui um vetor multidimensional(16) para cada palavra, a esse vetor é colocado pesos a cada frase
    tf.keras.layers.GlobalAveragePooling1D(),# diminui a dimensao de cada vetor para um unico
    tf.keras.layers.Dense(24, activation='relu'),# camada densa com 24 neuronios, zerando os vsores negativos
    tf.keras.layers.Dense(1, activation='sigmoid')# saida, retorn um valor entre 0 e 1
])
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
# ajusta a funcao de loss, otimizador e metrica de avaliacao

/home/beuren/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
E0000 00:00:1789158743.920854  263343 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [16]:
epochs = 30
history = model.fit(training_padded, training_labels, epochs=epochs, validation_data=(testing_padded, testing_labels), verbose=2)
# faz o treino do modelo

Epoch 1/30
625/625 - 2s - 4ms/step - accuracy: 0.5644 - loss: 0.6835 - val_accuracy: 0.5633 - val_loss: 0.6752
Epoch 2/30
625/625 - 2s - 3ms/step - accuracy: 0.6714 - loss: 0.6079 - val_accuracy: 0.8058 - val_loss: 0.5045
Epoch 3/30
625/625 - 2s - 3ms/step - accuracy: 0.8014 - loss: 0.4509 - val_accuracy: 0.8116 - val_loss: 0.4257
Epoch 4/30
625/625 - 2s - 3ms/step - accuracy: 0.8374 - loss: 0.3785 - val_accuracy: 0.8331 - val_loss: 0.3897
Epoch 5/30
625/625 - 2s - 2ms/step - accuracy: 0.8535 - loss: 0.3444 - val_accuracy: 0.7962 - val_loss: 0.4235
Epoch 6/30
625/625 - 2s - 3ms/step - accuracy: 0.8671 - loss: 0.3132 - val_accuracy: 0.8290 - val_loss: 0.3756
Epoch 7/30
625/625 - 2s - 3ms/step - accuracy: 0.8796 - loss: 0.2891 - val_accuracy: 0.8131 - val_loss: 0.4017
Epoch 8/30
625/625 - 2s - 3ms/step - accuracy: 0.8877 - loss: 0.2720 - val_accuracy: 0.8313 - val_loss: 0.3722
Epoch 9/30
625/625 - 2s - 3ms/step - accuracy: 0.8974 - loss: 0.2493 - val_accuracy: 0.8541 - val_loss: 0.3422
E

In [17]:
sentence = ["granny starting to fear spiders in the garden might be real", "game of thrones season finale showing this sunday night"]

In [18]:
sequence = tokenizer.texts_to_sequences(sentence)
padded = pad_sequences(sequence, padding='post', maxlen=120, truncating='post')
# transforma a frase em sequencia de indices e preenche com zeros

In [19]:
pred = model.predict(padded) 
# teste com uma predição
print(1 if pred[0]>0.5 else 0)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1


In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(10000,16, input_length=120),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
    # a camada Bidirecional permite um neuronio guarde um esstado de uma palavra para ao final da montagem da frase a rede de valor para cada palavra
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [29]:
sentence = [
    "local man finally finishes reading terms and conditions, ages 15 years",
    "senate votes to approve new infrastructure spending bill"
]

In [30]:
sequence = tokenizer.texts_to_sequences(sentence)
padded = pad_sequences(sequence, padding='post', maxlen=120, truncating='post')
# transforma a frase em sequencia de indices e preenche com zeros

In [31]:
pred.shape

(1, 1)

In [33]:
pred = model.predict(padded) 
# teste com uma predição
print(1 if pred[0]>0.5 else 0)
print(1 if pred[1]>0.5 else 0)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1
1
